In [0]:
# This Cell is sued to get Secrets we have in Key Vaultss

client_id = dbutils.secrets.get(scope="kv-scope", key="db-secret-client-id-app-reg")
client_secret = dbutils.secrets.get(scope="kv-scope", key="db-secret-value-appregi")
tenant_id = dbutils.secrets.get(scope="kv-scope", key="db-secret-tenant")

# Storage account name
storage_account = "stdehealthcareanalytics"

# OAuth configs
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
# SQL Server connection

sql_server = "healthcare-project-server-2026.database.windows.net"
sql_user = "username"
sql_pass = dbutils.secrets.get(scope="kv-scope", key="sql-pwd-azureportal")
sql_db = "healthcarebootcamp"

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_db}"

connection_properties = {
    "user": sql_user,
    "password": sql_pass,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
from pyspark.sql.functions import col

# Read FilesTable
files_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.FilesTable",
    properties=connection_properties
)

# Filter medications files
pending_files_df = files_df.filter(
    (col("Status") == "Bronze_Processed") &
    (col("FileName").like("medications%"))
)

# Create batch list
batch_list = [row.FileName for row in pending_files_df.select("FileName").collect()]

print("Files to process:", batch_list)

# 🔥 READ FROM FOLDER (NOT FILE NAME)
bronze_path = "abfss://bronze@stdehealthcareanalytics.dfs.core.windows.net/medications"

df = spark.read.format("parquet").load(bronze_path)

display(df.limit(10))

Files to process: []


medication_id,patient_id,visit_id,drug_name,dosage,start_date,end_date
RXN-17-0000001,PAT-15-0001665,VIS-15-0066822,Amlodipine,10mg,2017-05-23,2017-07-13
RXN-17-0000002,PAT-15-0002056,VIS-15-0059501,Metformin,25mg,2018-08-12,2018-10-17
RXN-17-0000003,PAT-15-0000186,VIS-15-0048230,Lisinopril,500mg,2018-04-26,2018-07-10
RXN-17-0000004,PAT-15-0000605,VIS-15-0082947,Amoxicillin,5mg,2018-12-28,2019-02-23
RXN-17-0000005,PAT-15-0004387,VIS-15-0037651,null,50mg,2019-07-25,2019-08-26
RXN-17-0000006,Â PAT-15-0004409,VIS-15-0017915,Amoxicillin,100mg,null,2019-10-27
RXN-17-0000007,PAT-15-0000252,VIS-15-0044620,Amoxicillin,25mg,2018-12-14,2018-12-22
null,Â PAT-15-0003375,Â VIS-15-0001994,Gabapentin,5mg,null,2019-03-10
Â RXN-17-0000009,PAT-15-0003343,VIS-15-0084535,Albuterol,10mg,2018-09-16,2018-11-08
null,Â PAT-15-0004838,NaN,Lisinopril,NaN,2019-10-03,2019-11-05


In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace

for file_name in batch_list:

    try:
        print("Cleaning:", file_name)

        # Read from bronze
        bronze_path = "abfss://bronze@stdehealthcareanalytics.dfs.core.windows.net/medications"
        df = spark.read.format("parquet").load(bronze_path)

        # Trim spaces
        for column in df.columns:
            df = df.withColumn(column, trim(col(column)))

        # Convert empty strings to NULL
        for column in df.columns:
            df = df.withColumn(column, when(col(column) == "", None).otherwise(col(column)))

        # 🔥 Replace IDs (ADJUST if names differ)
        df = df.withColumn("medication_id", regexp_replace(col("medication_id"), "[^A-Za-z0-9-]", ""))
        df = df.withColumn("patient_id", regexp_replace(col("patient_id"), "[^A-Za-z0-9-]", ""))

        # Replace "NULL" string with actual NULL
        df = df.replace("NULL", None)

        print("Cleaned:", file_name)
        display(df.limit(10))

    except Exception as e:
        print("Failed:", file_name, str(e))

In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace

# Trim all string columns
for column in df.columns:
    df = df.withColumn(column, trim(col(column)))

# Empty → NULL
for column in df.columns:
    df = df.withColumn(column, when(col(column) == "", None).otherwise(col(column)))

# Remove weird characters
for column in df.columns:
    df = df.withColumn(column,
        when(col(column).isNotNull(),
             regexp_replace(col(column).cast("string"), "Â|\t", "")
        ).otherwise(col(column))
    )

# "NULL" → NULL
df = df.replace("NULL", None)

display(df.limit(10))

medication_id,patient_id,visit_id,drug_name,dosage,start_date,end_date
RXN-17-0000001,PAT-15-0001665,VIS-15-0066822,Amlodipine,10mg,2017-05-23,2017-07-13
RXN-17-0000002,PAT-15-0002056,VIS-15-0059501,Metformin,25mg,2018-08-12,2018-10-17
RXN-17-0000003,PAT-15-0000186,VIS-15-0048230,Lisinopril,500mg,2018-04-26,2018-07-10
RXN-17-0000004,PAT-15-0000605,VIS-15-0082947,Amoxicillin,5mg,2018-12-28,2019-02-23
RXN-17-0000005,PAT-15-0004387,VIS-15-0037651,null,50mg,2019-07-25,2019-08-26
RXN-17-0000006,PAT-15-0004409,VIS-15-0017915,Amoxicillin,100mg,null,2019-10-27
RXN-17-0000007,PAT-15-0000252,VIS-15-0044620,Amoxicillin,25mg,2018-12-14,2018-12-22
null,PAT-15-0003375,VIS-15-0001994,Gabapentin,5mg,null,2019-03-10
RXN-17-0000009,PAT-15-0003343,VIS-15-0084535,Albuterol,10mg,2018-09-16,2018-11-08
null,PAT-15-0004838,NaN,Lisinopril,NaN,2019-10-03,2019-11-05


In [0]:
from pyspark.sql.functions import col

print("Writing to Silver...")

# Remove NULL PK
df_valid = df.filter(col("medication_id").isNotNull())

# Deduplicate
df_valid = df_valid.dropDuplicates(["medication_id"])

# Silver path
silver_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/medications/silver"

# Write to Silver
df_valid.write.format("delta").mode("append").save(f"{silver_path}/medications_silver")

print("Write complete")

Writing to Silver...
Write complete


In [0]:
df.display()

medication_id,patient_id,visit_id,drug_name,dosage,start_date,end_date
RXN-17-0000001,PAT-15-0001665,VIS-15-0066822,Amlodipine,10mg,2017-05-23,2017-07-13
RXN-17-0000002,PAT-15-0002056,VIS-15-0059501,Metformin,25mg,2018-08-12,2018-10-17
RXN-17-0000003,PAT-15-0000186,VIS-15-0048230,Lisinopril,500mg,2018-04-26,2018-07-10
RXN-17-0000004,PAT-15-0000605,VIS-15-0082947,Amoxicillin,5mg,2018-12-28,2019-02-23
RXN-17-0000005,PAT-15-0004387,VIS-15-0037651,null,50mg,2019-07-25,2019-08-26
RXN-17-0000006,PAT-15-0004409,VIS-15-0017915,Amoxicillin,100mg,null,2019-10-27
RXN-17-0000007,PAT-15-0000252,VIS-15-0044620,Amoxicillin,25mg,2018-12-14,2018-12-22
null,PAT-15-0003375,VIS-15-0001994,Gabapentin,5mg,null,2019-03-10
RXN-17-0000009,PAT-15-0003343,VIS-15-0084535,Albuterol,10mg,2018-09-16,2018-11-08
null,PAT-15-0004838,NaN,Lisinopril,NaN,2019-10-03,2019-11-05


In [0]:
from pyspark.sql.functions import col

# Identify bad records
bad_df = df.filter(
    col("medication_id").isNull() |
    col("patient_id").isNull()
)



# Path
bad_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/medications/badrecords"

# Write bad records
bad_df.write.format("delta").mode("append").save(bad_path)

display(bad_df)

medication_id,patient_id,visit_id,drug_name,dosage,start_date,end_date
RXN-15-0000006,null,VIS-15-0025810,Albuterol,25mg,2015-02-08,2015-04-10
null,PAT-15-0004589,VIS-15-0053377,Amoxicillin,10mg,2016-07-02,2016-08-19
null,null,VIS-15-0016690,Amlodipine,5mg,2016-01-09,2016-04-07
RXN-15-0000018,null,VIS-15-0005728,Lisinopril,100mg,2016-09-23,2016-10-07
null,PAT-15-0004300,VIS-15-0011180,Amlodipine,25mg,2016-03-16,2016-06-10
RXN-15-0000029,null,VIS-15-0033241,Albuterol,25mg,2016-02-25,2016-05-22
RXN-15-0000032,null,VIS-15-0093891,null,100mg,2017-11-22,2018-01-29
RXN-15-0000049,null,VIS-15-0069426,Atorvastatin,5mg,2016-04-15,2016-05-19
null,PAT-15-0000423,VIS-15-0002380,null,5mg,2015-11-15,2015-12-18
RXN-15-0000052,null,null,null,5mg,2016-07-28,2016-10-22
